# ✍️ Handwritten Character Recognition — CNN
## CodeAlpha Machine Learning Internship — Task 3
**Author:** Rose Sharma  
**Objective:** Identify handwritten digits using CNN on MNIST dataset  
**Model:** Convolutional Neural Network (CNN)  

---

## 📦 Step 1 — Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import confusion_matrix, classification_report

print(f'✅ TensorFlow version: {tf.__version__}')
print(f'✅ Keras version: {keras.__version__}')

## 📂 Step 2 — Load MNIST Dataset

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print('✅ MNIST Dataset loaded!')
print(f'   Train: {X_train.shape} | Labels: {y_train.shape}')
print(f'   Test : {X_test.shape}  | Labels: {y_test.shape}')
print(f'   Classes: {np.unique(y_train)} (digits 0-9)')
print(f'   Image size: 28x28 pixels, grayscale')

## 🔍 Step 3 — EDA & Visualization

In [ ]:
# Show sample images
fig, axes = plt.subplots(4, 10, figsize=(15, 7))
fig.suptitle('MNIST Sample Images — All 10 Digit Classes', fontsize=14, fontweight='bold')

for digit in range(10):
    indices = np.where(y_train == digit)[0][:4]
    for row, idx in enumerate(indices):
        axes[row, digit].imshow(X_train[idx], cmap='gray')
        axes[row, digit].axis('off')
        if row == 0:
            axes[row, digit].set_title(f'Digit {digit}', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('sample_digits.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Distribution', fontsize=14, fontweight='bold')

unique, counts = np.unique(y_train, return_counts=True)
axes[0].bar(unique, counts, color='#1a3c6e', alpha=0.85, edgecolor='white')
axes[0].set_title('Training Set')
axes[0].set_xlabel('Digit Class')
axes[0].set_ylabel('Count')
for i, (u, c) in enumerate(zip(unique, counts)):
    axes[0].text(u, c+100, str(c), ha='center', fontsize=9)

unique_t, counts_t = np.unique(y_test, return_counts=True)
axes[1].bar(unique_t, counts_t, color='#27ae60', alpha=0.85, edgecolor='white')
axes[1].set_title('Test Set')
axes[1].set_xlabel('Digit Class')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Dataset is well balanced across all 10 classes!')

## 🔧 Step 4 — Preprocessing

In [ ]:
# Normalize pixel values 0-255 → 0-1
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

# Reshape for CNN: (samples, height, width, channels)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test, 10)

print('✅ Preprocessing complete!')
print(f'   X_train shape: {X_train.shape}')
print(f'   X_test shape : {X_test.shape}')
print(f'   Pixel range  : [{X_train.min():.1f}, {X_train.max():.1f}]')
print(f'   Label shape  : {y_train_cat.shape}')

## 🧠 Step 5 — Build CNN Model

In [ ]:
def build_cnn():
    model = keras.Sequential([
        # Input
        layers.Input(shape=(28, 28, 1)),

        # Block 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Classifier
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_cnn()
model.summary()

## 🏋️ Step 6 — Train Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train, y_train_cat,
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Training complete!')

## 📊 Step 7 — Evaluate & Visualize

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'\n✅ Test Accuracy : {test_acc*100:.2f}%')
print(f'   Test Loss    : {test_loss:.4f}')

In [ ]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'], color='#1a3c6e', lw=2, label='Train')
axes[0].plot(history.history['val_accuracy'], color='#27ae60', lw=2, label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].axhline(y=test_acc, color='red', linestyle='--',
                label=f'Test: {test_acc:.4f}')
axes[0].legend()

axes[1].plot(history.history['loss'], color='#1a3c6e', lw=2, label='Train')
axes[1].plot(history.history['val_loss'], color='#27ae60', lw=2, label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix
y_pred_prob = model.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('CNN Confusion Matrix — MNIST Digits', fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print(classification_report(y_test, y_pred))

In [ ]:
# Show predictions on test samples
fig, axes = plt.subplots(3, 10, figsize=(18, 6))
fig.suptitle('CNN Predictions on Test Samples', fontsize=14, fontweight='bold')

for col in range(10):
    idx_class = np.where(y_test == col)[0]
    for row in range(3):
        idx = idx_class[row]
        axes[row, col].imshow(X_test[idx].reshape(28,28), cmap='gray')
        axes[row, col].axis('off')
        pred = y_pred[idx]
        color = 'green' if pred == y_test[idx] else 'red'
        axes[row, col].set_title(f'P:{pred}', fontsize=9,
                                  color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n✅ Green title = Correct | Red title = Wrong')

In [ ]:
# Save model
model.save('model/handwritten_cnn_model.h5')
print('✅ Model saved!')
print(f'\n🏆 FINAL RESULT')
print(f'   Test Accuracy: {test_acc*100:.2f}%')
print(f'   Test Loss    : {test_loss:.4f}')
print(f'   Author: Rose Sharma | CodeAlpha ML Internship — Task 3')